In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from mne.viz import plot_topomap

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Subject·Frequency × Channel·Time ICA on Wavelet Power

## Scope

This notebook prepares the data and runs the ICA decomposition on the
`(S × F, C × T)` reshape of the 4-D wavelet power tensor.  Subjects
and frequencies form the observation axis; channels and time are
combined into the feature axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)  — 4-D wavelet power
Reshape: (n_subjects × n_freqs,  n_channels × n_times)
         ──── observations ────  ──────── features ────────
```

Before reshaping the tensor is **z-scored along the time axis** so that
every `(subject, channel, frequency)` slice has zero mean and unit
variance.

ICA components live in the `C × T` feature space, so each component
reshapes back to `(C, T)` — a **spatiotemporal pattern** (channel ×
time) shared across subjects and frequencies.  Scores live on the
`(S, F)` axis, so each subject has their own per-frequency weighting
on every component.

## Pipeline

1. Load wavelet power for the chosen condition / music type (cached).
2. Apply subject / channel / time subsets for fast iteration.
3. Z-score along time and reshape to `(S×F, C×T)`.
4. PCA dimensionality reduction.
5. FastICA on the PCA scores.

After the final cell the following variables are available:

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X` | `(S×F, C×T)` | Z-scored reshaped 2-D matrix |
| `pca` | — | Fitted PCA object |
| `pca_scores` | `(S×F, K_pca)` | PCA-transformed scores |
| `ica` | — | Fitted FastICA object |
| `ica_scores` | `(S×F, K_ica)` | ICA scores (per-observation weights) |
| `ica_components` | `(K_ica, C×T)` | ICA spatiotemporal component patterns |
| `scores_2d` | `(S, F, K)` | ICA scores reshaped to subject × frequency |
| `components_2d` | `(K, C, T)` | ICA components reshaped to channel × time |

## Configuration

In [ ]:
# ── Experiment configuration ────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ─────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ───────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ─────────────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── Decomposition settings ──────────────────────────────────────────────
USE_PCA = True  # set False to run FastICA directly on the (S*F, C*T) matrix
N_COMPONENTS_PCA = 50  # number of PCA components to retain (ignored when USE_PCA=False)
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42  # reproducibility

# ── Storage directory ─────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "channel_time"
    / "pca_ica"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"Use PCA                : {USE_PCA}")
if USE_PCA:
    print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.

In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time
series to zero mean and unit variance.

**Reshaping** combines subjects and frequencies into the observation
axis, and channels and time into the feature axis:

```
(S, C, F, T)  →  transpose to  (S, F, C, T)
              →  reshape to     (S × F,  C × T)
                                obs.    features
```

Each row of the resulting 2-D matrix is the z-scored power surface
across all channels and time points for a single
`(subject, frequency)` pair.  PCA/ICA will discover **spatiotemporal
patterns** — channel × time fingerprints — shared across subjects and
frequencies.

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Reshape: (S, C, F, T) → transpose → (S, F, C, T) → (S*F, C*T)
bb_z_sf = bb_z.transpose(0, 2, 1, 3)  # (S, F, C, T)
n_obs = n_subjects * n_freqs
n_feat = n_channels * n_times
X = bb_z_sf.reshape(n_obs, n_feat)  # (S*F, C*T)

print(f"Reshaped matrix shape : {X.shape}")
print(f"  Observations (S×F)  : {X.shape[0]}")
print(f"  Features     (C×T)  : {X.shape[1]}")
print(f"Row means  ≈ 0 : {X.mean(axis=1).mean():.6f}")
print(f"Row stds        : {X.std(axis=1).mean():.4f}")

---
## Step 2 — PCA Dimensionality Reduction + ICA Decomposition

We first reduce the `C × T` feature space to `N_COMPONENTS_PCA`
principal components.  FastICA then rotates the PCA subspace to
maximise statistical independence, yielding `N_COMPONENTS_ICA`
independent spatiotemporal components.

**Results:**

| Object | Shape | Description |
|--------|-------|-------------|
| `ica_scores` | `(S×F, K)` | Per-observation weight for each IC |
| `ica_components` | `(K, C×T)` | Spatiotemporal pattern of each IC |
| `scores_2d` | `(S, F, K)` | ICA scores reshaped to subject × frequency |
| `components_2d` | `(K, C, T)` | ICA components reshaped to channel × time |

In [ ]:
# --- PCA (optional) ---
if USE_PCA:
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
    pca_scores = pca.fit_transform(X)  # (n_obs, K_pca)

    explained = pca.explained_variance_ratio_
    cumulative = np.cumsum(explained)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
    axes[0].set_xlabel("Component")
    axes[0].set_ylabel("Variance explained")
    axes[0].set_title(f"PCA Scree Plot — {LABEL}")

    axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
    axes[1].axhline(0.9, ls="--", color="gray", label="90%")
    axes[1].set_xlabel("Number of components")
    axes[1].set_ylabel("Cumulative variance explained")
    axes[1].set_title(f"Cumulative Variance — {LABEL}")
    axes[1].legend()

    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")

    print(
        f"Top {N_COMPONENTS_PCA} components explain "
        f"{cumulative[-1] * 100:.1f}% of total variance."
    )
else:
    print("PCA skipped — FastICA will be fit directly on X.")

# --- ICA ---
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
if USE_PCA:
    ica_scores = ica.fit_transform(pca_scores)  # (S*F, K_ica)
    ica_components = ica.components_ @ pca.components_  # (K_ica, C*T)
else:
    ica_scores = ica.fit_transform(X)  # (S*F, K_ica)
    ica_components = ica.components_  # (K_ica, C*T)

# Reshape ICA scores to (S, F, K)
scores_2d = ica_scores.reshape(n_subjects, n_freqs, N_COMPONENTS_ICA)  # (S, F, K)

# Reshape ICA components to (K, C, T) for spatiotemporal analysis
components_2d = ica_components.reshape(
    N_COMPONENTS_ICA, n_channels, n_times
)  # (K, C, T)

print(f"ICA scores shape       : {ica_scores.shape}")
print(f"ICA components shape   : {ica_components.shape}")
print(f"Scores 2-D shape       : {scores_2d.shape}  (S, F, K)")
print(f"Components 2-D shape   : {components_2d.shape}  (K, C, T)")

---
## Analysis (a) — Intersubject Correlation Matrix

For each ICA component we compute a **subject × subject** Pearson
correlation matrix.  Each subject is represented by their
**frequency profile** for that component — a row of `scores_2d[:, :, k]`
of length `F`.

High off-diagonal correlations indicate that the component captures a
consistent frequency activation pattern across individuals — a
hallmark of a stimulus-driven mode.

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Each row of scores_2d[:, :, i] is one subject's frequency profile (length F);
    # np.corrcoef rows-as-variables gives the (S, S) inter-subject correlation.
    corr_mat = np.corrcoef(scores_2d[:, :, i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Intersubject Correlation of IC Frequency Profiles — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "isc_component_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (b) — Mean Subject Loading per Component

For each ICA component, compute the **mean absolute score** per
subject.  Scores have shape `(S, F, K)`, so taking the mean of
absolute values over `F` gives a scalar per `(subject, component)`
pair — a summary of how strongly each participant expresses the
spatiotemporal mode across frequencies.

Subjects with uniformly high loadings indicate a stimulus-driven mode;
uneven loadings reflect individual differences.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `subject_loadings` | `(S, K)` | Mean `|score|` over frequencies per subject and IC |

In [ ]:
# Subject loadings: mean |score| over frequencies per subject
subject_loadings = np.abs(scores_2d).mean(axis=1)  # (S, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.barh(
        range(n_subjects),
        subject_loadings[:, i],
        color="darkorange",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_xlabel("|score|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Subject")
fig.suptitle(
    f"Per-Subject Mean Loading per Component — {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_subject_loadings.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (c) — Time–Frequency Map per Component (Outer Product)

For each ICA component we build a **frequency × time** map directly
from the ICA decomposition:

```
A = scores_2d        # (S, F, K)   — per-(subject, frequency) score weights
S = components_2d    # (K, C, T)   — per-component channel × time pattern

freq_profile[k]  = A[:, :, k].mean(axis=0)         # (F,)  subject-averaged score
time_profile[k]  = S[k].mean(axis=0)               # (T,)  channel-averaged activation
tf_map[k]        = outer(freq_profile[k], time_profile[k])  # (F, T)
```

The outer product gives a rank-1 approximation of the component's
frequency–time structure grounded entirely in the ICA solution.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `freq_profiles` | `(F, K)` | Subject-averaged ICA score per frequency |
| `time_profiles` | `(K, T)` | Channel-averaged IC temporal activation |
| `ft_maps` | `(K, F, T)` | Outer-product frequency × time map per component |

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)

# Frequency profile: collapse subjects from ICA scores  (S, F, K) → (F, K)
freq_profiles = scores_2d.mean(axis=0)  # (F, K)

# Time profile: average channels from ICA components  (K, C, T) → (K, T)
time_profiles = components_2d.mean(axis=1)  # (K, T)

# Outer product for each component: (F, K) x (K, T) → (K, F, T)
ft_maps = np.einsum("fk,kt->kft", freq_profiles, time_profiles)  # (K, F, T)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_maps[i]  # (F, T)
    vmin_s, vmax_s = np.percentile(data_i, 1), np.percentile(data_i, 99)
    mesh = ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="inferno",
        vmin=vmin_s,
        vmax=vmax_s,
        shading="auto",
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} — Freq × Time Map (outer product)", fontsize=10)
    fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Frequency × Time Maps per IC — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_time_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (d) — Mean & Variance Component Channel Loading (Topomap)

The spatiotemporal component already contains a channel axis, so the
**channel profile** of each component is obtained by averaging
`components_2d[k]` over time:

```
S = components_2d                                  # (K, C, T)
chan_profile[c, k]   = mean_t  components_2d[k, c, t]            # (C, K)
```

This profile is shared across subjects.  To produce a per-subject
topomap we scale it by the subject's overall (frequency-averaged)
score on each component:

```
A = scores_2d                                      # (S, F, K)
subj_score[s, k]     = mean_f  scores_2d[s, f, k]                # (S, K)
topo_per_subj[k,s,c] = subj_score[s, k] * chan_profile[c, k]     # (K, S, C)
```

The figure has two rows:

* **Row 1** — mean across subjects (`RdBu_r`, symmetric around zero):
  the average spatial fingerprint of each component.
* **Row 2** — variance across subjects (`viridis`, vmin=0): channels
  whose loading is consistent across subjects appear dark, while
  channels with high cross-subject divergence appear bright.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `chan_profile` | `(C, K)` | Time-averaged channel profile per IC (from `components_2d`) |
| `subj_score` | `(S, K)` | Per-subject scalar score per IC (mean over F) |
| `topo_per_subj` | `(K, S, C)` | Per-subject outer-product topomap per IC |
| `ica_ch_mean` | `(C, K)` | Mean topomap across subjects |
| `ica_ch_var` | `(C, K)` | Across-subject variance topomap |

In [ ]:
# Time-averaged channel profile from components → (C, K)
chan_profile = components_2d.mean(axis=2).T  # (C, K)

# Per-subject score per IC (mean over frequencies) → (S, K)
subj_score = scores_2d.mean(axis=1)  # (S, K)

# Per-subject topomap via outer product:
#   topo_per_subj[k, s, c] = subj_score[s, k] * chan_profile[c, k]
topo_per_subj = np.einsum("sk,ck->ksc", subj_score, chan_profile)  # (K, S, C)

# Mean and variance across subjects — equal weight per subject; transpose to (C, K)
ica_ch_mean = topo_per_subj.mean(axis=1).T  # (C, K)
ica_ch_var = topo_per_subj.var(axis=1).T  # (C, K)

# Get MNE Info for topomap (restrict to the channel subset used here)
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = min(6, N_COMPONENTS_ICA)

# Symmetric color limits around zero for the mean row
_vlim_mean = np.percentile(np.abs(ica_ch_mean[:, :n_show]), 99)
# Sequential color limits (variance is non-negative) shared across the row
_vmax_var = np.percentile(ica_ch_var[:, :n_show], 99)

fig, axes = plt.subplots(2, n_show, figsize=(3.5 * n_show, 7.5))
if n_show == 1:
    axes = axes.reshape(2, 1)

im_mean = None
im_var = None
for i in range(n_show):
    im_mean, _ = plot_topomap(
        ica_ch_mean[:, i],
        info,
        axes=axes[0, i],
        show=False,
        cmap="RdBu_r",
        vlim=(-_vlim_mean, _vlim_mean),
    )
    axes[0, i].set_title(f"IC {i + 1}", fontsize=10)

    im_var, _ = plot_topomap(
        ica_ch_var[:, i],
        info,
        axes=axes[1, i],
        show=False,
        cmap="viridis",
        vlim=(0, _vmax_var),
    )

axes[0, 0].set_ylabel("Mean", fontsize=11)
axes[1, 0].set_ylabel("Variance", fontsize=11)

fig.suptitle(
    f"Component Channel Loading (topomap) — Mean & Across-Subject Variance — {LABEL}",
    fontsize=12,
)
plt.colorbar(im_mean, ax=axes[0, -1], label="mean loading")
plt.colorbar(im_var, ax=axes[1, -1], label="variance")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_topomap_mean.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (e) — Subject × Frequency Loading Heatmap per Component

Scores in this reshape live directly in `(S, F, K)` — there is no
channel axis to collapse — so the per-IC subject × frequency loading
map is read off the scores tensor directly:

```
sf_loadings[s, f, k] = scores_2d[s, f, k]   # (S, F, K)
```

Each component is plotted as a heatmap with **subjects on the y-axis**
and **frequency on the x-axis**, so rows reveal each participant's
spectral fingerprint and columns reveal which frequencies are
consistently expressed across subjects.  A diverging colormap
(`RdBu_r`, symmetric around zero) preserves the sign of the ICA
loadings.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `sf_loadings` | `(S, F, K)` | ICA score per (subject, freq, IC) — alias of `scores_2d` |

In [ ]:
# Scores already live in (S, F, K) — no axis to collapse
sf_loadings = scores_2d  # alias for clarity

n_show = min(6, N_COMPONENTS_ICA)

# Symmetric color limits around zero, shared across the displayed components
_vlim_sf = np.percentile(np.abs(sf_loadings[:, :, :n_show]), 99)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.6 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

mesh = None
for i, ax in enumerate(axes):
    mesh = ax.pcolormesh(
        FREQS,
        np.arange(n_subjects),
        sf_loadings[:, :, i],
        cmap="RdBu_r",
        vmin=-_vlim_sf,
        vmax=_vlim_sf,
        shading="auto",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_ylabel("Subject")
    ax.set_title(f"IC {i + 1} — Subject × Frequency Loading", fontsize=10)
    fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025, label="loading")

axes[-1].set_xlabel("Frequency (Hz)")
fig.suptitle(
    f"Subject × Frequency Loading Heatmaps per IC — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_subject_frequency_heatmap.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (f) — Component Time Courses

Each ICA component lives in `(C, T)` space.  Collapsing the channel
axis by averaging gives the **channel-averaged temporal signature** of
each component — a single per-IC waveform that captures the shared
temporal pattern after spatial mixing has been integrated out:

```
time_profiles[k, t] = mean_c  components_2d[k, c, t]   # (K, T)
```

| Quantity | Shape | Description |
|----------|-------|-------------|
| `time_profiles` | `(K, T)` | Channel-averaged ICA temporal pattern per IC |

In [ ]:
# Channel-averaged temporal profile per component
time_profiles = components_2d.mean(axis=1)  # (K, T)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.2 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, time_profiles[i], lw=0.8, color="darkorange")
    ax.axhline(0.0, color="gray", lw=0.5, ls="--")
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"Component {i + 1} — Channel-Averaged Temporal Pattern", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"ICA Component Time Courses (channel-averaged) — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_component_timecourses.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (g) — Subject × Time Activation Heatmap per Component

Following the outer-product idea of Analysis (c) but pivoting from the
*frequency* axis to the *subject* axis, we build a **subject × time**
map per IC purely from the ICA outputs:

```
A = scores_2d                                   # (S, F, K)
S = components_2d                               # (K, C, T)

subj_profile[s, k]  = mean_f  scores_2d[s, f, k]                       # (S, K)
time_profile[k, t]  = mean_c  components_2d[k, c, t]                   # (K, T)
st_map[k, s, t]     = subj_profile[s, k] * time_profile[k, t]          # (K, S, T)
```

Each subject's activation is the channel-averaged temporal pattern
scaled by their freq-averaged loading.  Plotted as a heatmap with
**subjects on the y-axis** and **time on the x-axis**.

A diverging colormap (`RdBu_r`, symmetric around zero) preserves sign,
with limits shared across the displayed components for cross-IC
comparability.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `subj_profile` | `(S, K)` | Freq-averaged ICA score per subject (alias of `subj_score`) |
| `st_maps` | `(K, S, T)` | Outer-product subject × time map per component |

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)

# Subject profile: collapse frequencies from ICA scores  (S, F, K) → (S, K)
subj_profile = scores_2d.mean(axis=1)  # (S, K)

# Time profile: channel-averaged components (already computed above)  (K, C, T) → (K, T)
time_profile = components_2d.mean(axis=1)  # (K, T)

# Outer product per component: (S, K) x (K, T) → (K, S, T)
st_maps = np.einsum("sk,kt->kst", subj_profile, time_profile)  # (K, S, T)

# Symmetric color limits around zero, shared across the displayed components
_vlim_st = np.percentile(np.abs(st_maps[:n_show]), 99)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.6 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

mesh = None
for i, ax in enumerate(axes):
    mesh = ax.pcolormesh(
        time,
        np.arange(n_subjects),
        st_maps[i],
        cmap="RdBu_r",
        vmin=-_vlim_st,
        vmax=_vlim_st,
        shading="auto",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_ylabel("Subject")
    ax.set_title(f"IC {i + 1} — Subject × Time Activation (outer product)", fontsize=10)
    fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025, label="activation")

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Subject × Time Activation Heatmaps per IC — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_subject_time_heatmap.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")